In [2]:
import rasterio
import numpy as np
from rasterio.enums import Resampling
import os

PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

# ── Load reference grid from master exclusion mask ─────────
with rasterio.open(f"{PROCESSED}/master_exclusion_mask.tif") as ref:
    REF_TRANSFORM = ref.transform
    REF_SHAPE     = (ref.height, ref.width)
    REF_CRS       = ref.crs
    REF_PROFILE   = ref.profile.copy()
    exclusion     = ref.read(1)

print(f"Reference grid: {REF_SHAPE} | CRS: {REF_CRS}")
print(f"Viable pixels: {(exclusion == 0).sum():,} of {exclusion.size:,}")

def load_resample(path, shape, label=""):
    """Load raster and resample to reference shape."""
    with rasterio.open(path) as src:
        data = src.read(
            1,
            out_shape=shape,
            resampling=Resampling.bilinear
        ).astype(np.float32)
        nodata = src.nodata
    if nodata is not None:
        data[data == nodata] = np.nan
    print(f"  ✅ {label}: min={np.nanmin(data):.2f} max={np.nanmax(data):.2f}")
    return data

# ── Load all feature layers ────────────────────────────────
print("\nLoading feature layers...")
dist_pop   = load_resample(f"{PROCESSED}/dist_pop.tif",   REF_SHAPE, "dist_pop (km)")
dist_port  = load_resample(f"{PROCESSED}/dist_port.tif",  REF_SHAPE, "dist_port (km)")
dist_road  = load_resample(f"{PROCESSED}/dist_road.tif",  REF_SHAPE, "dist_road (km)")
dist_river = load_resample(f"{PROCESSED}/dist_river.tif", REF_SHAPE, "dist_river (km)")
pop_dens   = load_resample(f"{PROCESSED}/population_density.tif", REF_SHAPE, "population density")

# ── MCDM Scoring (0–100) ───────────────────────────────────
print("\nCalculating suitability scores...")

score = np.zeros(REF_SHAPE, dtype=np.float32)
mask  = (exclusion == 0)  # only score viable pixels

# ── SAFETY (30 pts) ────────────────────────────────────────
# All non-excluded pixels pass safety — exclusion already applied
score[mask] += 30
print(f"  Safety (30pts): applied to {mask.sum():,} viable pixels")

# ── DEMAND PROXIMITY (30 pts) ──────────────────────────────
# Distance to population centers
score[mask & (dist_pop < 5)]                          += 30
score[mask & (dist_pop >= 5)  & (dist_pop < 10)]     += 20
score[mask & (dist_pop >= 10) & (dist_pop < 20)]     += 10
score[mask & (dist_pop >= 20) & (dist_pop < 40)]     += 5
print(f"  Demand proximity (30pts): done")

# ── LOGISTICS / PORT ACCESS (20 pts) ──────────────────────
# Framed as island logistics feasibility proxy
score[mask & (dist_port < 10)]                        += 20
score[mask & (dist_port >= 10) & (dist_port < 25)]   += 12
score[mask & (dist_port >= 25) & (dist_port < 50)]   += 6
print(f"  Port access/logistics (20pts): done")

# ── INFRASTRUCTURE / ROADS (10 pts) ───────────────────────
score[mask & (dist_road < 5)]                         += 10
score[mask & (dist_road >= 5) & (dist_road < 15)]    += 5
print(f"  Road access (10pts): done")

# ── FRESHWATER (10 pts) ────────────────────────────────────
score[mask & (dist_river < 2)]                        += 10
score[mask & (dist_river >= 2) & (dist_river < 5)]   += 5
print(f"  Freshwater access (10pts): done")

# ── Apply exclusion mask — set excluded to 0 ──────────────
score[exclusion == 1] = 0

# ── Statistics ────────────────────────────────────────────
viable_scores = score[mask]
print(f"\nSuitability score statistics (viable areas only):")
print(f"  Min:    {viable_scores.min():.1f}")
print(f"  Max:    {viable_scores.max():.1f}")
print(f"  Mean:   {viable_scores.mean():.1f}")
print(f"  Median: {np.median(viable_scores):.1f}")
print(f"\nScore distribution:")
for threshold in [30, 50, 60, 70, 80, 90]:
    n = (viable_scores >= threshold).sum()
    pct = n / viable_scores.size * 100
    print(f"  Score >= {threshold}: {n:,} pixels ({pct:.2f}%)")

# ── Save suitability raster ────────────────────────────────
profile = REF_PROFILE.copy()
profile.update({
    "dtype":    "float32",
    "count":    1,
    "nodata":   -9999,
    "compress": "lzw",
    "tiled":    True,
    "blockxsize": 256,
    "blockysize": 256,
    "bigtiff":  "IF_SAFER"
})
score[exclusion == 1] = -9999

out_path = f"{PROCESSED}/suitability_scores.tif"
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(score, 1)

size = os.path.getsize(out_path) / 1024 / 1024
print(f"\n✅ Suitability scores saved: {size:.1f} MB")

Reference grid: (61200, 39600) | CRS: EPSG:4326
Viable pixels: 1,750,857,760 of 2,423,520,000

Loading feature layers...
  ✅ dist_pop (km): min=0.00 max=584.90
  ✅ dist_port (km): min=0.00 max=597.82
  ✅ dist_road (km): min=0.01 max=578.32
  ✅ dist_river (km): min=0.00 max=576.79
  ✅ population density: min=0.00 max=9465.68

Calculating suitability scores...
  Safety (30pts): applied to 1,750,857,760 viable pixels
  Demand proximity (30pts): done
  Port access/logistics (20pts): done
  Road access (10pts): done
  Freshwater access (10pts): done

Suitability score statistics (viable areas only):
  Min:    30.0
  Max:    100.0
  Mean:   42.1
  Median: 30.0

Score distribution:
  Score >= 30: 1,750,857,760 pixels (100.00%)
  Score >= 50: 364,187,668 pixels (20.80%)
  Score >= 60: 310,294,140 pixels (17.72%)
  Score >= 70: 288,650,701 pixels (16.49%)
  Score >= 80: 270,140,364 pixels (15.43%)
  Score >= 90: 250,552,426 pixels (14.31%)

✅ Suitability scores saved: 102.2 MB


In [1]:
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

print("Extracting top candidate sites (row-by-row)...")

results = []

with rasterio.open(f"{PROCESSED}/suitability_scores.tif") as src:
    transform = src.transform
    height    = src.height
    width     = src.width

    for row in range(height):
        score_row = src.read(1, window=rasterio.windows.Window(0, row, width, 1))[0]

        if not np.any(score_row >= 70):
            continue

        cols     = np.arange(width)
        lons_row = transform.c + cols * transform.a
        lat      = transform.f + row  * transform.e

        valid    = score_row >= 70
        v_scores = score_row[valid]
        v_lons   = lons_row[valid]

        # Use int32 instead of int16
        g_lon = (v_lons // 0.1).astype(np.int32)
        g_lat = np.int32(lat // 0.1)
        g_id  = g_lat * 10000 + g_lon

        for s, lo, gi in zip(v_scores, v_lons, g_id):
            results.append((float(s), float(lo), float(lat), int(gi)))

        if row % 5000 == 0:
            print(f"  Row {row}/{height} | candidates so far: {len(results):,}")

print(f"\n  Total valid pixels: {len(results):,}")

df = pd.DataFrame(results, columns=["score", "lon", "lat", "grid_id"])
del results

idx_best = df.groupby("grid_id")["score"].idxmax()
thinned  = df.loc[idx_best].reset_index(drop=True)
del df

print(f"  After thinning: {len(thinned):,} candidate sites")

top_200 = thinned.sort_values("score", ascending=False).head(200).reset_index(drop=True)
top_200["site_id"] = [f"SITE_{i+1:03d}" for i in range(len(top_200))]

print(f"\nScore distribution:")
print(f"  Min:  {top_200['score'].min():.1f}")
print(f"  Max:  {top_200['score'].max():.1f}")
print(f"  Mean: {top_200['score'].mean():.1f}")

print(f"\nTop 10 sites:")
print(top_200[["site_id", "lat", "lon", "score"]].head(10).to_string())

top_200_gdf = gpd.GeoDataFrame(
    top_200,
    geometry=[Point(r.lon, r.lat) for r in top_200.itertuples()],
    crs="EPSG:4326"
)
top_200_gdf.to_file(f"{PROCESSED}/top_200_candidates.gpkg", driver="GPKG")
top_200[["site_id", "lat", "lon", "score"]].to_csv(
    f"{PROCESSED}/top_200_candidates.csv", index=False
)
print(f"\n✅ Saved: top_200_candidates.gpkg + .csv")

Extracting top candidate sites (row-by-row)...
  Row 15000/61200 | candidates so far: 1,137,615
  Row 20000/61200 | candidates so far: 6,438,246
  Row 25000/61200 | candidates so far: 12,958,026
  Row 30000/61200 | candidates so far: 20,069,000
  Row 35000/61200 | candidates so far: 27,650,062
  Row 40000/61200 | candidates so far: 36,607,304
  Row 45000/61200 | candidates so far: 41,645,625
  Row 50000/61200 | candidates so far: 53,457,504
  Row 55000/61200 | candidates so far: 76,432,057
  Row 60000/61200 | candidates so far: 242,179,731

  Total valid pixels: 288,650,701
  After thinning: 4,608 candidate sites

Score distribution:
  Min:  100.0
  Max:  100.0
  Mean: 100.0

Top 10 sites:
    site_id        lat         lon  score
0  SITE_001  10.999861  124.500694  100.0
1  SITE_002   5.462361  125.340694  100.0
2  SITE_003   5.569306  120.848472  100.0
3  SITE_004   5.545694  120.785139  100.0
4  SITE_005   5.599861  125.451806  100.0
5  SITE_006   5.599861  125.315972  100.0
6  SITE

In [1]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

# ── Target: 1km grid (~0.009 deg) ─────────────────────────
# Philippines bbox: lon 114–128, lat 4–22
# At 1km: ~1556 cols x ~2000 rows — very manageable
TARGET_H = 2000
TARGET_W = 1600

print(f"Working at {TARGET_W}x{TARGET_H} (approx 1km resolution)")

def load_1km(path):
    with rasterio.open(path) as src:
        return src.read(
            1,
            out_shape=(TARGET_H, TARGET_W),
            resampling=Resampling.bilinear
        ).astype(np.float32)

print("Loading layers at 1km...")
exclusion  = load_1km(f"{PROCESSED}/master_exclusion_mask.tif")
dist_pop   = load_1km(f"{PROCESSED}/dist_pop.tif")
dist_port  = load_1km(f"{PROCESSED}/dist_port.tif")
dist_road  = load_1km(f"{PROCESSED}/dist_road.tif")
dist_river = load_1km(f"{PROCESSED}/dist_river.tif")
dem        = load_1km(f"{PROCESSED}/dem_4326.tif")
print("  ✅ All layers loaded")

# ── Derive slope ────────────────────────────────────────────
# At 1km grid, pixel size ~1km
dy, dx = np.gradient(dem.astype(np.float64), 1000, 1000)
slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2))).astype(np.float32)
del dem, dx, dy
print("  ✅ Slope derived")

# ── Score ───────────────────────────────────────────────────
mask  = (exclusion < 0.5)  # viable pixels
score = np.zeros((TARGET_H, TARGET_W), dtype=np.float32)

# Safety base
score[mask] += 30

# Demand proximity — exponential decay
score[mask] += (30 * np.exp(-np.clip(dist_pop,   0, 100) / 15))[mask]

# Port logistics — exponential decay
score[mask] += (20 * np.exp(-np.clip(dist_port,  0, 200) / 30))[mask]

# Road access
score[mask] += (10 * np.exp(-np.clip(dist_road,  0, 100) / 10))[mask]

# Freshwater
score[mask] += (10 * np.exp(-np.clip(dist_river, 0,  50) /  5))[mask]

# Slope penalty — steep land gets reduced score
slope_penalty = np.clip(slope / 15, 0, 1) * 10
score[mask] -= slope_penalty[mask]
score = np.clip(score, 0, 100)

# Normalize viable scores to 0–100
s_min = score[mask].min()
s_max = score[mask].max()
score[mask] = (score[mask] - s_min) / (s_max - s_min) * 100
score[~mask] = -9999

print(f"\nScore statistics:")
print(f"  Min:    {score[mask].min():.1f}")
print(f"  Max:    {score[mask].max():.1f}")
print(f"  Mean:   {score[mask].mean():.1f}")
print(f"  Median: {np.median(score[mask]):.1f}")
for t in [50, 60, 70, 80, 90]:
    n = (score[mask] >= t).sum()
    print(f"  >= {t}: {n:,} pixels ({n/mask.sum()*100:.2f}%)")

# ── Save 1km suitability raster ─────────────────────────────
with rasterio.open(f"{PROCESSED}/master_exclusion_mask.tif") as ref:
    bounds = ref.bounds

from rasterio.transform import from_bounds as fb
transform_1km = fb(bounds.left, bounds.bottom,
                   bounds.right, bounds.top,
                   TARGET_W, TARGET_H)

profile_1km = {
    "driver": "GTiff", "dtype": "float32", "count": 1,
    "height": TARGET_H, "width": TARGET_W,
    "transform": transform_1km, "crs": "EPSG:4326",
    "nodata": -9999, "compress": "lzw",
    "tiled": True, "blockxsize": 256, "blockysize": 256
}
with rasterio.open(f"{PROCESSED}/suitability_1km.tif", "w", **profile_1km) as dst:
    dst.write(score, 1)
print("\n✅ suitability_1km.tif saved")

# ── Extract top 200 sites ────────────────────────────────────
rows, cols = np.where(score >= 60)
lons = bounds.left + (cols + 0.5) * (bounds.right  - bounds.left) / TARGET_W
lats = bounds.top  - (rows + 0.5) * (bounds.top    - bounds.bottom) / TARGET_H
scores = score[rows, cols]

df = pd.DataFrame({
    "score": scores,
    "lon":   lons,
    "lat":   lats,
    "grid_lon": (lons // 0.5).astype(int),
    "grid_lat": (lats // 0.5).astype(int),
})

# Best per 0.5° cell
thinned = df.loc[df.groupby(["grid_lon","grid_lat"])["score"].idxmax()].reset_index(drop=True)
top_200 = thinned.sort_values("score", ascending=False).head(200).reset_index(drop=True)
top_200["site_id"] = [f"SITE_{i+1:03d}" for i in range(len(top_200))]

print(f"\nAfter thinning: {len(thinned):,} sites")
print(f"\nTop 10 sites:")
print(top_200[["site_id","lat","lon","score"]].head(10).to_string())

top_200_gdf = gpd.GeoDataFrame(
    top_200,
    geometry=[Point(r.lon, r.lat) for r in top_200.itertuples()],
    crs="EPSG:4326"
)
top_200_gdf.to_file(f"{PROCESSED}/top_200_candidates.gpkg", driver="GPKG")
top_200[["site_id","lat","lon","score"]].to_csv(
    f"{PROCESSED}/top_200_candidates.csv", index=False
)
print("\n✅ Top 200 candidates saved")

Working at 1600x2000 (approx 1km resolution)
Loading layers at 1km...
  ✅ All layers loaded
  ✅ Slope derived

Score statistics:
  Min:    0.0
  Max:    100.0
  Mean:   18.8
  Median: 13.4
  >= 50: 108,227 pixels (4.70%)
  >= 60: 77,580 pixels (3.37%)
  >= 70: 50,311 pixels (2.19%)
  >= 80: 26,229 pixels (1.14%)
  >= 90: 4,767 pixels (0.21%)

✅ suitability_1km.tif saved

After thinning: 219 sites

Top 10 sites:
    site_id        lat         lon       score
0  SITE_001  14.289389  121.510174  100.000000
1  SITE_002  13.932389  121.627049   99.153282
2  SITE_003  10.710889  122.568924   99.129471
3  SITE_004   9.716389  123.902674   98.618744
4  SITE_005  12.300389  125.023299   98.605095
5  SITE_006  14.994889  120.595799   98.261459
6  SITE_007   5.959389  121.015174   98.139244
7  SITE_008  10.183889  125.559549   97.959450
8  SITE_009  12.410889  122.012049   97.920135
9  SITE_010  15.003389  120.595799   97.912270

✅ Top 200 candidates saved


In [2]:
import geopandas as gpd

RAW = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\raw"

# Load province boundaries
adm2 = gpd.read_file(
    f"{RAW}/phl_adm_psa_namria_20231106_shp/phl_admbnda_adm2_psa_namria_20231106.shp"
)

# Spatial join to get province names
top_200_gdf = gpd.read_file(f"{PROCESSED}/top_200_candidates.gpkg")
joined = gpd.sjoin(
    top_200_gdf,
    adm2[["ADM2_EN", "ADM1_EN", "geometry"]],
    how="left",
    predicate="within"
)

# Clean up
joined = joined.drop(columns=["index_right"])
joined = joined.rename(columns={"ADM2_EN": "province", "ADM1_EN": "region"})

print("Top 10 sites with locations:")
print(joined[["site_id","lat","lon","score","province","region"]].head(10).to_string())

# Save enriched
joined.to_file(f"{PROCESSED}/top_200_candidates.gpkg", driver="GPKG")
joined[["site_id","lat","lon","score","province","region"]].to_csv(
    f"{PROCESSED}/top_200_candidates.csv", index=False
)
print("\n✅ Sites enriched with province and region names")

Top 10 sites with locations:
    site_id        lat         lon       score         province                                                   region
0  SITE_001  14.289389  121.510174  100.000000           Laguna                                 Region IV-A (Calabarzon)
1  SITE_002  13.932389  121.627049   99.153282           Quezon                                 Region IV-A (Calabarzon)
2  SITE_003  10.710889  122.568924   99.129471           Iloilo                              Region VI (Western Visayas)
3  SITE_004   9.716389  123.902674   98.618744            Bohol                             Region VII (Central Visayas)
4  SITE_005  12.300389  125.023299   98.605095   Northern Samar                            Region VIII (Eastern Visayas)
5  SITE_006  14.994889  120.595799   98.261459         Pampanga                               Region III (Central Luzon)
6  SITE_007   5.959389  121.015174   98.139244             Sulu  Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)
7  